# SkillVerify — Colab training notebook
Train a lightweight **skill-level evidence classifier** from technical evidence text.

**Labels:** `beginner`, `intermediate`, `advanced`

The notebook:
1. installs dependencies
2. accepts a real CSV (`text,label`) or creates a small synthetic bootstrap dataset
3. splits and evaluates a TF-IDF + Logistic Regression model
4. prints a confusion matrix and classification report
5. exports `skill_model.joblib`
6. shows how to run inference on new GitHub evidence

> The synthetic dataset is only for demonstrating the pipeline. For a real verification product, replace it with labeled assessment evidence collected from consented users and human/assessment-grounded labels.

In [ ]:
!pip -q install -U scikit-learn pandas matplotlib seaborn joblib

In [ ]:
import io, os, random, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
print("Ready")

## 1. Load labeled evidence

Upload a CSV with exactly two useful columns: `text` and `label`. The text should describe observable engineering evidence (for example, code structure, tests, API usage, CI configuration).

In [ ]:
from google.colab import files

uploaded = files.upload()
if uploaded:
    filename = next(iter(uploaded))
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print("Loaded:", filename)
    print(df.head())
else:
    df = None
print("Use columns:", None if df is None else df.columns.tolist())

In [ ]:
# If you do not have labeled data yet, generate a bootstrap dataset.
# Replace this section with real labels before using the model for actual decisions.

if df is None:
    beginner_templates = [
        "small script with one function and no tests; uses basic Python syntax",
        "simple CRUD example with minimal validation and no automated tests",
        "basic React component with local state and little error handling",
        "simple SQL select queries without joins or transaction handling",
        "small API call example with hardcoded values and no retry logic",
    ]
    intermediate_templates = [
        "multi-module Python service with validation, exceptions, tests and REST endpoints",
        "React application with reusable components, hooks, API integration and loading states",
        "SQL application using joins, indexes, transactions and parameterized queries",
        "backend service with authentication, logging, unit tests and structured error handling",
        "CI workflow running linting, tests and build steps for a production-like project",
    ]
    advanced_templates = [
        "distributed service with caching, observability, concurrency control and integration tests",
        "complex React application with state architecture, performance optimization and robust testing",
        "database design with query optimization, migrations, indexing strategy and transaction boundaries",
        "backend system with queues, rate limiting, security controls, profiling and comprehensive tests",
        "production deployment with CI/CD, containers, monitoring, rollback strategy and infrastructure automation",
    ]
    rows=[]
    for label, templates in [
        ("beginner", beginner_templates),
        ("intermediate", intermediate_templates),
        ("advanced", advanced_templates),
    ]:
        for _ in range(80):
            base=random.choice(templates)
            rows.append({"text": base, "label": label})
    df=pd.DataFrame(rows)

df=df[["text","label"]].dropna()
df["text"]=df["text"].astype(str).str.strip()
df["label"]=df["label"].astype(str).str.lower().str.strip()
df=df[df["label"].isin(["beginner","intermediate","advanced"])]
print(df["label"].value_counts())
print("Rows:", len(df))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=1,
        max_features=20000,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Skill-level classifier")
plt.show()

## 2. Export the trained model
Copy the resulting `skill_model.joblib` into `backend/` of the SkillVerify project and set:

`SKILL_MODEL_PATH=backend/skill_model.joblib`

The app treats this classifier as a **secondary signal**; Gemini remains the main evidence synthesis layer.

In [ ]:
MODEL_PATH = "skill_model.joblib"
joblib.dump(model, MODEL_PATH)
print("Saved:", MODEL_PATH, os.path.getsize(MODEL_PATH), "bytes")

files.download(MODEL_PATH)

## 3. Test inference on new evidence

In [ ]:
examples = [
    "repository has pytest coverage, typed Python modules, REST endpoints, CI checks, caching and structured error handling",
    "small Python script with a few functions and no tests",
    "React app with hooks, reusable components, API integration and automated tests"
]
probs = model.predict_proba(examples)
preds = model.predict(examples)

for text, pred, prob in zip(examples, preds, probs):
    confidence = float(np.max(prob))
    print(f"{pred:12s} confidence={confidence:.2f} :: {text}")

## 4. Suggested real-data schema

Build a CSV like:

```csv
text,label
"pytest coverage + typed service + CI + REST API",intermediate
"distributed worker + caching + profiling + integration tests",advanced
"single script with no tests",beginner
```

For production-quality verification, collect multiple independent evidence snippets per person/skill, use assessment results as labels, remove personally identifying data, and evaluate the model on a held-out set from users it never saw during training.